# RPA Bot Status Prediction

## 📌 What is this notebook about?

We will **predict the status of RPA (Robotic Process Automation) bots** using machine learning.

Each bot can be in one of 4 states:
- `Running` — actively working
- `Idle` — waiting for tasks
- `Maintenance` — under repair/update
- `Deprecated` — no longer in use

We use features like **tasks per day, error rate, success rate, AI assistance**, etc. to predict the bot's status.

> ⚠️ **Key Insight (discovered during EDA):** The `Idle` and `Running` classes have **nearly identical numeric distributions** — this is a real-world data challenge, not a model bug.

### 📁 Dataset Files
| File | Rows | Description |
|------|------|-------------|
| `software_bots.csv` | 200,000 | Bot-level features — main dataset |
| `rpa_companies.csv` | 5,000 | Company info |
| `automation_projects.csv` | 50,000 | Project-level info |

### 🔑 Steps We Follow
1. Import Libraries
2. Load & Explore Data (EDA)
3. Preprocess Data
4. Feature Engineering
5. Train Random Forest Model
6. Evaluate the Model
7. Feature Importance
8. Conclusion

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries imported successfully!')

## Step 2: Load and Explore the Data

In [ ]:
bots_df      = pd.read_csv('/kaggle/input/software-bots/software_bots.csv')
companies_df = pd.read_csv('/kaggle/input/software-bots/rpa_companies.csv')
projects_df  = pd.read_csv('/kaggle/input/software-bots/automation_projects.csv')

print('software_bots      :', bots_df.shape)
print('rpa_companies      :', companies_df.shape)
print('automation_projects:', projects_df.shape)

In [ ]:
bots_df.head()

In [ ]:
# Check for missing values and data types
print('Missing values:')
print(bots_df.isnull().sum())
print('\nData types:')
print(bots_df.dtypes)

In [ ]:
# Numeric summary statistics
bots_df.describe().round(2)

In [ ]:
# --- Plot 1: Target variable distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = bots_df['bot_status'].value_counts()
sns.barplot(x=status_counts.index, y=status_counts.values, palette='Set2', ax=axes[0])
axes[0].set_title('Bot Status Distribution', fontsize=14)
axes[0].set_xlabel('Bot Status')
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, status_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontsize=11)

axes[1].pie(status_counts.values, labels=status_counts.index,
            autopct='%1.1f%%', colors=sns.color_palette('Set2'), startangle=90)
axes[1].set_title('Bot Status Share (%)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Error Rate vs Success Rate per status ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=bots_df, x='bot_status', y='error_rate_percent', palette='Set2', ax=axes[0])
axes[0].set_title('Error Rate by Bot Status')

sns.boxplot(data=bots_df, x='bot_status', y='success_rate_percent', palette='Set3', ax=axes[1])
axes[1].set_title('Success Rate by Bot Status')

plt.tight_layout()
plt.show()

# KEY INSIGHT: Idle and Running have almost identical distributions!
print('Mean stats per status:')
print(bots_df.groupby('bot_status')[['error_rate_percent','success_rate_percent']].mean().round(2))

In [ ]:
# --- Plot 3: Correlation heatmap ---
numeric_cols = ['tasks_per_day', 'average_execution_time_seconds',
                'error_rate_percent', 'success_rate_percent']

plt.figure(figsize=(8, 5))
sns.heatmap(bots_df[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

## Step 3: Preprocess the Data

ML models need **numbers only**. We:
1. Drop ID/name columns (not useful)
2. Convert Yes/No → 1/0
3. Label-encode text columns
4. Encode the target `bot_status`

In [ ]:
df = bots_df.copy()

# Drop ID and date columns
df.drop(columns=['bot_id', 'project_id', 'bot_name', 'deployment_date'], inplace=True)

# Yes/No → 1/0
binary_cols = ['ai_assisted', 'machine_learning_enabled',
               'unattended_bot', 'attended_bot', 'cloud_hosted']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Encode text columns
le = LabelEncoder()
for col in ['bot_category', 'operating_region']:
    df[col] = le.fit_transform(df[col])

# Encode target
df['bot_status'] = le.fit_transform(df['bot_status'])
status_classes = le.classes_
print('Classes:', list(status_classes))
print('Shape after preprocessing:', df.shape)

## Step 4: Feature Engineering

We create new features to help the model learn better patterns:
- `error_success_ratio` — ratio of errors to success
- `tasks_per_exec_time` — throughput metric
- `ai_ml_combo` — bot uses both AI + ML

In [ ]:
# Error-to-success ratio (high = poor performing bot)
df['error_success_ratio'] = df['error_rate_percent'] / (df['success_rate_percent'] + 1e-5)

# Throughput: tasks done per second of execution time
df['tasks_per_exec_time'] = df['tasks_per_day'] / (df['average_execution_time_seconds'] + 1e-5)

# Is the bot using both AI and ML capabilities?
df['ai_ml_combo'] = df['ai_assisted'] * df['machine_learning_enabled']

print('New features added. Final shape:', df.shape)

## Step 5: Train/Test Split

- **80%** for training
- **20%** for testing

`stratify=y` ensures each class is proportionally represented in both sets.

In [ ]:
X = df.drop(columns=['bot_status'])
y = df['bot_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing  samples : {X_test.shape[0]:,}')

## Step 6: Train Random Forest Classifier

**Random Forest** builds many decision trees and takes a majority vote.
It handles mixed feature types well and is robust to noise.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,     # 200 decision trees
    max_depth=20,         # max depth per tree
    min_samples_leaf=5,   # min samples at leaf node
    n_jobs=-1,            # use all CPU cores
    random_state=42
)

print('Training Random Forest... (may take ~30 seconds)')
rf_model.fit(X_train, y_train)
print('Model trained!')

## Step 7: Evaluate the Model

In [ ]:
y_pred = rf_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc * 100:.2f}%')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=status_classes))

In [ ]:
# --- Why is Idle recall low? ---
# Idle and Running bots have nearly identical numeric features.
# This is a data-level challenge — the dataset does not contain
# features that clearly separate Idle from Running bots.
print('Idle vs Running feature comparison:')
for col in ['error_rate_percent', 'success_rate_percent', 'tasks_per_day']:
    idle_mean = bots_df[bots_df['bot_status'] == 'Idle'][col].mean()
    run_mean  = bots_df[bots_df['bot_status'] == 'Running'][col].mean()
    print(f'  {col}: Idle={idle_mean:.2f} | Running={run_mean:.2f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=status_classes)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8: Feature Importance

Which features did the model rely on the most?

In [ ]:
importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importance — Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print('Top 5 Features:')
print(importances.head())

## Conclusion

We built a **multi-class classification model** to predict the status of RPA software bots.

### What We Did
| Step | Action |
|------|--------|
| EDA | Explored 200,000 bot records; found Idle ≈ Running in numeric features |
| Preprocessing | Encoded Yes/No columns and categorical text columns |
| Feature Engineering | Added error/success ratio, throughput, AI+ML combo flag |
| Model | Random Forest — 200 trees, max depth 20 |
| Accuracy | ~70% — honest ceiling given data overlap between Idle & Running |

### Key Findings
- **`success_rate_percent` and `error_rate_percent`** are the top two features, accounting for ~80% of model importance.
- **`Deprecated` bots are predicted perfectly** (99% precision) — they have much higher error rates.
- **`Idle` vs `Running`** is the hard case — they have statistically identical distributions across all numeric features, which is the main reason accuracy is capped at ~70%.

### What Could Be Improved
- Merge `automation_projects.csv` to add project-level features (budget, ROI, robots deployed)
- Try **XGBoost** or **LightGBM** for faster and often better results
- Use **SHAP** values for model explainability
- Explore whether time-based features from `deployment_date` add signal

> If you found this notebook helpful, please **upvote ⬆️** and feel free to fork and experiment!